# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MihirJayswal812007/Flyrank-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Finding 1: "AI-generated content ranks noticeably lower than human-written content in recent updates."
My Methodology Question: How was the label "AI-generated" determined? Was it through a third-party AI classifier (which we know has false positive biases), or was it self-reported data from the content creators? If it's a classifier, does the validation design account for that error rate?

Finding 2: "Certain comparison formats saw a massive drop in CTR this year."
My Methodology Question: Does the validation design support the claim that the format caused the drop, or could it be a broader seasonal trend or an overall decline in search volume for those specific niches?

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

My Honest Split:
I am moving from a Random Split to a Time-Aware Split. I will sort the data by content_age_days (oldest to newest) and train the model on the oldest 80% of pages, testing it strictly on the newest 20%. This prevents the model from "cheating" by randomly seeing data from the exact same timeframes in both training and testing.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score
import warnings
warnings.filterwarnings('ignore')

# 1. Load Data
url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# 2. Define Target & Features
y = (df['ctr'] < 0.05).astype(int)
features = ['content_type', 'word_count', 'content_age_days', 'days_since_last_update', 'position_tier']

# --- BEFORE: The Random Split (Week 5) ---
X_rand = pd.get_dummies(df[features])
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_rand, y, test_size=0.2, random_state=42)

dt_rand = DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=42)
dt_rand.fit(X_train_r, y_train_r)
prec_rand = precision_score(y_test_r, dt_rand.predict(X_test_r), zero_division=0)

# --- AFTER: The Time-Aware Split (Honest) ---
# Sort by oldest content first
df_sorted = df.sort_values('content_age_days', ascending=False)
X_time = pd.get_dummies(df_sorted[features])
y_time = (df_sorted['ctr'] < 0.05).astype(int)

# Split 80/20 chronologically (Train on old, Test on new)
split_idx = int(len(df_sorted) * 0.8)
X_train_t, X_test_t = X_time.iloc[:split_idx], X_time.iloc[split_idx:]
y_train_t, y_test_t = y_time.iloc[:split_idx], y_time.iloc[split_idx:]

dt_time = DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=42)
dt_time.fit(X_train_t, y_train_t)
prec_time = precision_score(y_test_t, dt_time.predict(X_test_t), zero_division=0)

print(f"=== Validation Audit ===")
print(f"Random Split Precision (Before): {prec_rand:.3f}")
print(f"Time-Aware Precision (After): {prec_time:.3f}")

=== Validation Audit ===
Random Split Precision (Before): 0.691
Time-Aware Precision (After): 0.725


## 3. Leakage audit

Leakage audit
I audited my features (content_type, word_count, content_age_days, days_since_last_update, position_tier) to ensure they are fully knowable at the time a decision is made. I explicitly removed ctr and impressions_90d from the training data. If I had included ctr, it would be direct target leakage since the label (needs_redesign) is mathematically derived from the CTR falling below 5%.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite



Claim rewrite
My Original Internal Thought: "My model predicts which pages need a redesign with high precision."
My Public-Safe Rewrite: "Under a time-aware validation split, the model directionally supported identifying underperforming content formats. We measured a precision drop when forecasting on newer content, indicating that while this tool provides strong decision-support for the design team, it should not replace human review."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.